In [ ]:
import torch

from base import CUDA, DTYPE
from FreeFermion.linalg import (RandPureCov, majorana_gram, wick, williamson)
from FreeFermion.cuda import fermion

## `FreeFermion.linalg`: sampling, Williamson's theorem, Wick/Pfaffian contraction

| function | what it returns |
|---|---|
| `RandPureCov(dim)` | random pure covariance `Gamma = q J q^T` of `dim` Majorana modes |
| `williamson(gamma)` | `(R, lambdas)` with `Gamma = R^T D R`, `D = diag(lambda_k J)` and `p_k = (1 - lambda_k) / 2` |
| `wick(covariance, operators)` | Gaussian expectation of an ordered Majorana product, by Wick's theorem |
| `majorana_gram(covariance)` | Gram matrix of the Majorana-excitation basis over the Majoranas of the covariance |
| `FreeFermion.cuda.fermion` | the batched Wick contraction chain (`operator_vectors`, `vacuum_contraction`, `pfaffian`) of `FreeFermion/cuda/fermion.cu` |


### 1. Sampling and the Williamson decomposition


In [2]:
A = RandPureCov(6)                    # pure covariance of 6 Majorana modes
R, lambdas = williamson(A)

D = torch.block_diag(*[lambdas[k] * torch.tensor([[0.0, -1.0], [1.0, 0.0]], dtype=A.dtype, device=CUDA)
                       for k in range(3)])   # the D = diag(lambda_k J) of the convention
print('|Gamma - R^T D R| =', float((A - R.T @ D @ R).abs().max()))
print('|R^T R - I|       =', float((R.T @ R - torch.eye(6, dtype=R.dtype, device=CUDA)).abs().max()))
print('lambdas           =', [round(float(x), 12) for x in lambdas])
print('D =')
print(D)


|Gamma - R^T D R| = 6.661338147750939e-16
|R^T R - I|       = 4.440892098500626e-16
lambdas           = [1.0, 1.0, 1.0]
D =
tensor([[ 0.0000, -1.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -1.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -1.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000]],
       device='cuda:0', dtype=torch.float64)


### 2. Normal modes


In [3]:
# a pure covariance: every lambda_k = 1, hence every mode is empty
occupations = (1.0 - lambdas) / 2.0                  # p_k of the convention
coeff = (R[0::2] + 1j * R[1::2]) / 2.0               # d_k over the 2n Majoranas

print('occupations p_k          =', [round(float(x), 12) for x in occupations])
print('|2 c c^dag - I|           =', float((2.0 * (coeff @ coeff.conj().T)
                                          - torch.eye(3, dtype=coeff.dtype, device=CUDA)).abs().max()))


occupations p_k          = [0.0, 0.0, -0.0]
|2 c c^dag - I|           = 4.441622981383784e-16


### 3. The Wick/Pfaffian chain (CUDA kernels)

The kernels below are the raw chain; section 5 wraps them in `wick`, which takes the covariance directly instead of the mode coefficients.


In [4]:
def by_pairings(matrix: torch.Tensor) -> torch.Tensor:
    '''
    brute-force Pfaffian by enumerating the pairings, only for the 4x4 cross-check.
    '''
    size = matrix.shape[0]
    if size == 0:
        return torch.ones((), dtype=matrix.dtype, device=matrix.device)
    total = torch.zeros((), dtype=matrix.dtype, device=matrix.device)
    for j in range(1, size):
        keep = [k for k in range(1, size) if k != j]
        total = total + (-1) ** (j + 1) * matrix[0, j] * by_pairings(matrix[keep][:, keep])
    return total

torch.manual_seed(42)
antisymmetric = torch.randn(4, 4, dtype=DTYPE, device=CUDA)
antisymmetric = antisymmetric - antisymmetric.T
print('kernel pfaffian =', complex(fermion.pfaffian(antisymmetric.unsqueeze(0))[0]))
print('by pairings     =', complex(by_pairings(antisymmetric)))


kernel pfaffian = (0.2157776435591134+2.1989921889955846j)
by pairings     = (0.21577764355911333+2.1989921889955846j)


In [5]:
# one empty mode: <0|gamma_0 gamma_1|0> = 1j
single = torch.tensor([[0.0, -1.0], [1.0, 0.0]], dtype=torch.float64, device=CUDA)
R, lambdas = williamson(single)
coeff = (R[0::2] + 1j * R[1::2]) / 2.0

# the kernels are batched, so a single operator list is a batch of one (kind 2 = gamma_mu)
codes = torch.tensor([[[2, 0], [2, 1]]], dtype=torch.long, device=CUDA)
vectors = fermion.operator_vectors(coeff, codes)      # (1, 2, 2)
matrices = fermion.vacuum_contraction(vectors, 1)     # (1, 2, 2)
values = fermion.pfaffian(matrices)                   # (1,)

print('gamma_0 in the mode basis =', vectors[0, 0].tolist())
print('<0|gamma_0 gamma_1|0>     =', complex(values[0]), '(expected 1j)')


gamma_0 in the mode basis = [(-1-0j), (-1+0j)]
<0|gamma_0 gamma_1|0>     = 1j (expected 1j)


### 4. Batched calls, which is what the kernels are for


In [6]:
# 8 identical operator lists [gamma_0, gamma_1], one call for the whole batch
codes = torch.tensor([[[2, 0], [2, 1]]] * 8, dtype=torch.long, device=CUDA)   # kind 2 = gamma_mu
vectors = fermion.operator_vectors(coeff, codes)      # (8, 2, 2)
matrices = fermion.vacuum_contraction(vectors, 1)     # (8, 2, 2)
values = fermion.pfaffian(matrices)                   # (8,)
print('shapes:', tuple(vectors.shape), tuple(matrices.shape), tuple(values.shape))
print('values:', values.tolist())


shapes: (8, 2, 2) (8, 2, 2) (8,)
values: [1j, 1j, 1j, 1j, 1j, 1j, 1j, 1j]


In [7]:
import time


def bench(fn, reps: int) -> float:
    '''
    mean wall time of one call of fn in microseconds.
    '''
    for _ in range(2):
        fn()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps * 1e6


many = torch.tensor([[[2, 0], [2, 1]]] * 256, dtype=torch.long, device=CUDA)
one = many[:1]
batched_us = bench(lambda: fermion.pfaffian(fermion.vacuum_contraction(
    fermion.operator_vectors(coeff, many), 1)), 20)
scalar_us = bench(lambda: [fermion.pfaffian(fermion.vacuum_contraction(
    fermion.operator_vectors(coeff, one), 1)) for _ in range(256)], 2)
print(f'batched, 256 lists : {batched_us:9.1f} us  ({batched_us / 256:.2f} us each)')
print(f'one by one, 256    : {scalar_us:9.1f} us  ({scalar_us / 256:.2f} us each)')
print(f'speedup            : {scalar_us / batched_us:.0f}x')


batched, 256 lists :      20.3 us  (0.08 us each)
one by one, 256    :    4468.8 us  (17.46 us each)
speedup            : 220x


### 5. `wick`: any ordered Majorana product

`wick(covariance, operators)` is the primitive the contractions of this module are built from: the Gaussian expectation of an ordered product of Majoranas, which Wick's theorem turns into the Pfaffian of the contraction matrix. The labels are 0-based Majorana indices of the covariance, a plain list is one product, and a rectangular batch gives a batch of values. A repeated Majorana is its own square, so it contracts with itself: the expectation of `gamma_a gamma_a` is 1, and no product has to be reduced first.


In [8]:
Gamma = RandPureCov(16)                    # pure covariance of 3 modes = 6 Majoranas
print('Gamma:', tuple(Gamma.shape))
print('<gamma_2 gamma_2>                     =', complex(wick(Gamma, [2, 2])), '  (self contraction, 1)')
print('<gamma_0 gamma_1>                     =', complex(wick(Gamma, [0, 1])))
print(' -1j * Gamma[0, 1]                    =', complex(-1j * Gamma[0, 1]))
print('<gamma_0 gamma_1 gamma_0 gamma_1>     =', complex(wick(Gamma, [0, 1, 0, 1])), '  (-1)')
print('odd <gamma_0 gamma_1 gamma_2>         =', complex(wick(Gamma, [0, 1, 2])), '  (0)')
print('empty product                         =', complex(wick(Gamma, [])), '  (1)')


Gamma: (16, 16)
<gamma_2 gamma_2>                     = (1+0j)   (self contraction, 1)
<gamma_0 gamma_1>                     = -0.46834159826074356j
 -1j * Gamma[0, 1]                    = -0.46834159826074356j
<gamma_0 gamma_1 gamma_0 gamma_1>     = (-1-0j)   (-1)
odd <gamma_0 gamma_1 gamma_2>         = 0j   (0)
empty product                         = (1+0j)   (1)


In [9]:
lists = [[0, 1], [2, 3], [4, 5], [1, 4]]         # a batch needs equal lengths
batched = wick(Gamma, lists)
print('batched   :', [complex(x) for x in batched])
print('one by one:', [complex(wick(Gamma, one)) for one in lists])
print('difference:', float((batched - torch.stack(
    [wick(Gamma, one) for one in lists])).abs().max()))


batched   : [-0.46834159826074356j, -0.013028940664766573j, -0.3946315176441095j, -0.039306182671226264j]
one by one: [-0.46834159826074356j, -0.013028940664766573j, -0.3946315176441095j, -0.039306182671226264j]
difference: 0.0


### 6. `majorana_gram`: the Gram of the excitation basis

The basis states are the products `gamma_S` applied to the Gaussian state, for every subset S of the Majoranas of the given covariance, so k Majoranas give 2**k states. Wick makes every overlap a Pfaffian of the contraction kernel `-i covariance`, and only the symmetric difference of the two subsets survives, so 2**k Pfaffians cover the whole matrix. The state enters only through the two point functions of these Majoranas, so the Majoranas of the basis are chosen by slicing the covariance: the slice is generally a mixed state, and that is a valid input. The diagonal is 1 (every basis state is normalised), the blocks of odd total excitation vanish, and the kernel of the Gram holds the linear dependence of the 2**k states.


In [10]:
sub = Gamma[:4, :4]                       # the Majoranas of the basis, chosen by slicing
G = majorana_gram(sub)
print('Gram:', tuple(G.shape), ' diagonal deviation:', float((torch.diagonal(G).real - 1).abs().max()))
print('|G - G^dag|:', float((G - G.conj().T).abs().max()))
sizes = torch.tensor([bin(m).count('1') for m in range(G.shape[0])], device=CUDA)
odd = ((sizes[:, None] + sizes[None, :]) % 2).bool()
print('odd total excitation block, max |G|:', float(G[odd].abs().max()) if bool(odd.any()) else 0.0)
print('spectrum:', [round(float(x), 6) for x in torch.linalg.eigvalsh(G)])


Gram: (16, 16)  diagonal deviation: 0.0
|G - G^dag|: 0.0
odd total excitation block, max |G|: 0.0
spectrum: [0.148089, 0.148089, 0.148089, 0.148089, 0.165946, 0.165946, 0.165946, 0.165946, 1.738185, 1.738185, 1.738185, 1.738185, 1.947781, 1.947781, 1.947781, 1.947781]


In [11]:
print('Majoranas  states  size  rank   nonzero spectrum')
for count in (2, 4, 6 , 8, 10, 12):
    G = majorana_gram(Gamma[:count, :count])
    ev = torch.linalg.eigvalsh(G)
    ev = ev[ev > 1e-9]
print(f'{count:>8} {G.shape[0]:>7} {G.shape[1]:>5} {int(ev.numel()):>5}   {[round(float(x), 4) for x in ev]}')


Majoranas  states  size  rank   nonzero spectrum
      12    4096  4096   256   [1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 1.8388, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755, 5.0755,